# IF25-40305: Sistem Teknologi Multimedia (STM)
## Modul 01 — Audio Signal Fundamentals: Sample Rate Conversion & Anti-Aliasing

**Institut Teknologi Sumatera (ITERA) — Program Studi Informatika**  
*Materi Kuliah & Hands-on Lab Semester Ganjil 2026/2027*

---

### 🎯 Capaian Pembelajaran (Learning Outcomes)
Setelah menyelesaikan modul hands-on ini, mahasiswa diharapkan mampu:
1. **Memahami Standar Laju Sampel ($f_s$) di Industri**: Mengetahui standar penggunaan laju sampel (8 kHz untuk telekomunikasi/telephony, 16 kHz standar emas AI model seperti Whisper, 44.1 kHz untuk Audio CD, dan 48 kHz untuk audio video/YouTube).
2. **Membedakan Pengubahan Metadata vs Resampling Hakiki**: Memahami secara langsung mengapa sekadar mengganti label `sr` pemutar menimbulkan efek *monster voice* atau *chipmunk voice*.
3. **Memahami Konsep Dasar Aliasing**: Mengamati fenomena *spectral fold-back* di mana frekuensi tinggi menyamar menjadi frekuensi rendah palsu saat laju sampel terlalu rendah.
4. **Menganalisis Mekanisme Downsampling (Desimasi $M$)**: Memahami pembuangan $(M-1)$ sampel secara berkala dan peran mutlak **Filter Anti-Aliasing (LPF)** sebelum desimasi.
5. **Menguasai Tahapan Upsampling (Interpolasi $L$)**: Memecah dan mengamati dua tahap sekuensial: *Zero-Stuffing* (penyisipan nol) dan *Reconstruction Low-Pass Filtering* dengan penguatan gain $L$.
6. **Menerapkan Resampling Rasio Pecahan ($L/M$)**: Mengonversi laju sampel non-kelipatan bulat seperti standar CD 44.1 kHz ke video 48 kHz ($160/147$).
7. **Mengamati dan Mendengarkan Aliasing Auditif**: Melakukan eksperimen auditif menggunakan *Chirp Signal* dan mengamati perbedaan spektrogram antara desimasi naif vs standar DSP.


## 1. Import Library & Persiapan Lingkungan

Pustaka yang digunakan dalam modul ini mencakup ekosistem saintifik dan pengolahan audio modern Python:
- `numpy`: Manipulasi array numerik sinyal dan fungsi matematis.
- `matplotlib.pyplot`: Visualisasi bentuk gelombang (*waveform*), spektrum FFT, dan spektrogram.
- `scipy.signal`: Pemrosesan sinyal digital (filter Butterworth IIR, polyphase resampling FIR, dan sintesis sinyal *chirp*).
- `librosa`: Ekstraksi audio, analisis MIR (*Music Information Retrieval*), dan resampling standar industri.
- `IPython.display`: Pemutar audio interaktif (*playback*) di dalam notebook.
- `soundfile`: Pembaca dan penyimpan berkas audio WAV/FLAC.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import librosa
import librosa.display
import IPython.display as ipd
from IPython.display import display, Audio
import soundfile as sf
import os

# Konfigurasi visualisasi agar tajam, rapi, dan konsisten
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("✅ Seluruh pustaka DSP dan Audio berhasil dimuat!")


## 2. Kebutuhan Konversi Laju Sampel (Standar Industri)

Di industri nyata, berbagai perangkat keras, platform media, dan model kecerdasan buatan bekerja pada laju sampel yang berbeda:

| Laju Sampel ($f_s$) | Standar Penggunaan di Lapangan |
|---|---|
| **8.000 Hz (8 kHz)** | Komunikasi suara telepon seluler tradisional & walkie-talkie (rentang pita vokal ~300–3400 Hz) |
| **16.000 Hz (16 kHz)** | **Standar emas AI** (Automatic Speech Recognition: OpenAI Whisper, Siri, Google Speech) |
| **44.100 Hz (44.1 kHz)** | Audio CD standar, rekaman musik komersial, streaming Spotify |
| **48.000 Hz (48 kHz)** | Audio video profesional, bioskop, siaran TV digital, YouTube |
| **96.000 Hz (96 kHz)** | Studio rekaman audio resolusi tinggi (*mastering*) |

> ⚠️ **Peringatan Penting: Resampling Bukan Sekadar Mengganti Angka Label `sr`!**  
> Mengubah laju sampel membutuhkan rekonstruksi titik-titik sampel secara matematis. Jika kita hanya mengganti label `sr` pemutar tanpa menghitung ulang titik sampelnya:  
> - Memutar audio dengan label `sr` **setengahnya** membuat audio diputar **2× lebih lambat dan pitch turun 1 oktaf (efek suara monster/raksasa)**.  
> - Memutar audio dengan label `sr` **dua kalinya** membuat audio diputar **2× lebih cepat dan pitch naik melengking (efek suara kartun *chipmunk*)**.


In [ ]:
# 1. Sintesis nada murni A4 (440 Hz) selama 2 detik pada laju standar CD (44.100 Hz)
fs_orig = 44100
duration = 2.0
t = np.linspace(0, duration, int(fs_orig * duration), endpoint=False)
f_tone = 440.0  # Frekuensi nada A4 (standar tala musik konser)
y_orig = 0.5 * np.sin(2 * np.pi * f_tone * t)

print(f"Total sampel asli : {len(y_orig)} sampel")
print(f"Laju sampel asli  : {fs_orig} Hz")
print(f"Durasi            : {duration} detik")

# Visualisasi waveform kecil (10 ms pertama) agar osilasi periodik terlihat jelas
fig, ax = plt.subplots(figsize=(8, 2))
t_preview = 0.01  # 10 milidetik
n_preview = int(fs_orig * t_preview)
ax.plot(t[:n_preview] * 1000, y_orig[:n_preview], color='#0284c7', lw=1.6)
ax.set_title("Waveform Nada A4 (440 Hz) — 10 ms Pertama (fs = 44.100 Hz)", fontsize=10, fontweight='bold')
ax.set_xlabel("Waktu (ms)", fontsize=9)
ax.set_ylabel("Amplitudo", fontsize=9)
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# Dengarkan nada asli A4 (440 Hz)
print("🎵 Nada Asli A4 (440 Hz, playback rate = 44.100 Hz):")
display(ipd.Audio(y_orig, rate=fs_orig))

# Demonstrasi kesalahan label pemutar:
print("👹 Efek Monster (Playback rate diatur ke 22.050 Hz — nada turun ke 220 Hz A3):")
display(ipd.Audio(y_orig, rate=22050))

print("🐿️ Efek Chipmunk (Playback rate diatur ke 88.200 Hz — nada melesat ke 880 Hz A5):")
display(ipd.Audio(y_orig, rate=88200))


### 🎙️ Demonstrasi Nyata pada Suara Manusia (*Speech Audio*)

Pada sinyal sinus murni di atas, perubahan frekuensi nada terdengar jelas. Namun, efek *chipmunk* dan *monster voice* akan terdengar **jauh lebih dramatis dan nyata** ketika diterapkan pada rekaman suara manusia (*speech*).

Mari kita muat sebuah rekaman suara manusia (durasi ~4 detik) dan dengarkan bagaimana manipulasi laju pemutaran mengubah artikulasi kata dan formant vokal manusia:


In [ ]:
# Mencari berkas audio speech secara cerdas (mendukung root atau folder notebook)
possible_paths = [
    'audio/speech_sample.wav',
    'notebook_handson/audio/speech_sample.wav',
    '../audio/speech_sample.wav'
]
audio_path = next((p for p in possible_paths if os.path.exists(p)), None)

if audio_path is None:
    print("Mengunduh sampel speech dari pustaka resmi LibriSpeech...")
    audio_path = librosa.ex('libri1')

# Muat cuplikan 4 detik dari rekaman suara manusia
y_speech, sr_speech = librosa.load(audio_path, sr=None, duration=4.0)

print(f"File sampel       : {audio_path}")
print(f"Laju sampel asli  : {sr_speech} Hz")
print(f"Total sampel data : {len(y_speech)} sampel ({len(y_speech)/sr_speech:.2f} detik)")

# Visualisasi waveform kecil rekaman suara manusia
fig, ax = plt.subplots(figsize=(8, 2))
t_speech = np.linspace(0, len(y_speech) / sr_speech, len(y_speech))
ax.plot(t_speech, y_speech, color='#334155', lw=0.6)
ax.set_title(f"Waveform Suara Manusia (Speech) — Durasi {len(y_speech)/sr_speech:.1f}s, fs = {sr_speech} Hz", fontsize=10, fontweight='bold')
ax.set_xlabel("Waktu (detik)", fontsize=9)
ax.set_ylabel("Amplitudo", fontsize=9)
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# 1. Suara Asli (Playback Normal)
print("1. 🗣️ Suara Asli Manusia (Normal, rate = 22.050 Hz):")
display(ipd.Audio(y_speech, rate=sr_speech))

# 2. Efek Monster (Playback 0.5x laju sampel)
sr_monster = int(sr_speech * 0.5)  # 11.025 Hz
print(f"2. 👹 Efek Suara Monster (Playback rate diatur ke {sr_monster} Hz — lambat & vokal berat):")
display(ipd.Audio(y_speech, rate=sr_monster))

# 3. Efek Chipmunk (Playback 2x laju sampel)
sr_chipmunk = int(sr_speech * 2.0)  # 44.100 Hz
print(f"3. 🐿️ Efek Suara Chipmunk (Playback rate diatur ke {sr_chipmunk} Hz — cepat & melengking tinggi):")
display(ipd.Audio(y_speech, rate=sr_chipmunk))


## 3. Memahami Konsep Dasar Aliasing (Fenomena Menyamar)

### ❓ Apa Itu Aliasing?
Kata **aliasing** berasal dari kata bahasa Inggris *"alias"* yang berarti **nama samaran**.

Dalam pengolahan sinyal digital:
> **Aliasing** adalah fenomena di mana suatu komponen frekuensi tinggi ($f > f_s / 2$) **menyamar menjadi frekuensi rendah palsu** setelah proses pencuplikan diskrit (*sampling*), karena titik-titik sampel diambil terlalu jarang untuk mencatat osilasi gelombang aslinya secara akurat.

### 🏎️ Analogi Dunia Nyata: Efek Roda Mobil (*Wagon-Wheel Effect*)
Pernahkah Anda melihat video rekaman mobil balap atau helikopter, di mana roda mobil atau bilah baling-baling tampak **berputar sangat lambat**, berhenti diam, atau bahkan **berputar terbalik ke belakang**?  
- **Penyebab:** Sensor kamera video merekam frame terlalu lambat (misal 30 FPS) dibandingkan kecepatan putaran roda (misal 32 rotasi/detik).
- Pada audio, gelombang suara yang bergetar lebih cepat dari kemampuan pencuplikan akan melipat balik (*spectral fold-back*) menjadi nada rendah yang sebenarnya tidak pernah ada di sinyal asli!

### 🔢 Formula Matematis Aliasing
Menurut **Teorema Nyquist-Shannon**, untuk merekam frekuensi tertinggi $f_{\text{max}}$ tanpa distorsi, laju cuplik harus memenuhi:
$$f_s \ge 2 \cdot f_{\text{max}} \quad \Longleftrightarrow \quad f_{\text{Nyquist}} = \frac{f_s}{2}$$

Jika suatu komponen sinyal memiliki frekuensi $f > \frac{f_s}{2}$, maka frekuensi tersebut akan melipat balik ke frekuensi semu:
$$f_{\text{alias}} = |f - k \cdot f_s|$$


In [ ]:
# Demonstrasi Aliasing dengan Bermain Array 10 Sampel Sederhana
fs_alias = 10  # Laju sampel sangat rendah: 10 Hz (Batas Nyquist = 5 Hz)
n_pts = 10
t_samples = np.arange(n_pts) / fs_alias  # Titik waktu: [0.0, 0.1, 0.2, ..., 0.9] detik

# Kita definisikan dua frekuensi berbeda:
f_low = 1.0   # 1 Hz  (< 5 Hz: Sinyal Aman di bawah batas Nyquist)
f_high = 9.0  # 9 Hz  (> 5 Hz: Melanggar batas Nyquist!)

# Hitung nilai sampel pada kedua sinyal kosinus:
x_low = np.cos(2 * np.pi * f_low * t_samples)
x_high = np.cos(2 * np.pi * f_high * t_samples)

print("=== INSPEKSI NILAI ARRAY DISKRIT ===")
print("Sampel x_low  (1 Hz) :", np.round(x_low, 3))
print("Sampel x_high (9 Hz) :", np.round(x_high, 3))
print(f"Selisih maksimum     : {np.max(np.abs(x_low - x_high)):.2e}")

if np.allclose(x_low, x_high):
    print("\n⚠️ PERHATIKAN: Kedua array bernilai 100% IDENTIK!")
    print("Sistem komputer TIDAK DAPAT MEMBEDAKAN apakah sinyal input adalah 1 Hz atau 9 Hz.")
    print("Gelombang 9 Hz telah sepenuhnya MENYAMAR (aliased) menjadi gelombang 1 Hz!")

# Visualisasi: Kurva kontinu vs titik-titik sampel
t_fine = np.linspace(0, 1.0, 1000)
c_low = np.cos(2 * np.pi * f_low * t_fine)
c_high = np.cos(2 * np.pi * f_high * t_fine)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t_fine, c_high, color='#ef4444', lw=1.2, alpha=0.7, linestyle='--', label=f'Sinyal Frekuensi Tinggi Asli (f = {f_high:.0f} Hz)')
ax.plot(t_fine, c_low, color='#0284c7', lw=2.0, label=f'Sinyal Frekuensi Rendah Palsu / Alias (f = {f_low:.0f} Hz)')
ax.scatter(t_samples, x_low, color='#0f172a', s=50, zorder=5, label=f'Titik Sampel Diambil (fs = {fs_alias} Hz)')

ax.set_title("Bukti Visual Aliasing: Titik Sampel 10 Hz Jatuh Tepat di Kedua Gelombang Sekaligus!", fontsize=10, fontweight='bold')
ax.set_xlabel("Waktu (detik)", fontsize=9)
ax.set_ylabel("Amplitudo", fontsize=9)
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()


## 4. Mekanisme Downsampling (Desimasi Faktor $M$)

**Desimasi** adalah proses mengurangi laju sampel dengan faktor bulat $M$:
$$f_{s2} = \frac{f_{s1}}{M}$$

### Cara Kerja:
- Ambil **1 sampel**, lalu buang **$(M-1)$ sampel** berikutnya secara teratur.
- Formula matematis: $y[m] = x[M \cdot m]$.
- Misal $M = 2$: simpan sampel indeks genap $x[0], x[2], x[4], \dots$ dan buang indeks ganjil $x[1], x[3], \dots$.

### ⚠️ Konsekuensi Kritis: Penurunan Batas Nyquist!
Saat laju sampel turun dari $f_{s1}$ menjadi $\frac{f_{s1}}{M}$, batas Nyquist baru ikut turun menjadi:
$$f_{\text{Nyquist baru}} = \frac{f_{s1}}{2M}$$

Jika sinyal asli mengandung komponen frekuensi $f > \frac{f_{s1}}{2M}$, komponen tersebut akan mengalami **aliasing parah**!

Mari kita buktikan dengan membuat sinyal komposit:
- Nada primer $500\text{ Hz}$ (amplitudo 0.5)
- Nada pengganggu $3.500\text{ Hz}$ (amplitudo 0.5)
- Laju awal $f_{s1} = 8.000\text{ Hz}$ (Batas Nyquist awal $= 4.000\text{ Hz}$).
- Desimasi $M = 2 \implies f_{s2} = 4.000\text{ Hz}$ (Batas Nyquist baru $= 2.000\text{ Hz}$).


In [ ]:
# Membuat sinyal komposit: Nada rendah (500 Hz) + Nada tinggi (3500 Hz)
fs_in = 8000
duration_test = 0.5
t_in = np.linspace(0, duration_test, int(fs_in * duration_test), endpoint=False)

signal_500 = 0.5 * np.sin(2 * np.pi * 500 * t_in)    # Sinyal primer
signal_3500 = 0.5 * np.sin(2 * np.pi * 3500 * t_in)  # Sinyal frekuensi tinggi
x = signal_500 + signal_3500

# Downsampling naif: memotong array dengan slicing [::2] tanpa filtering
M = 2
y_naive = x[::M]
fs_out = fs_in // M  # 4000 Hz

print(f"Sampel awal        : {len(x)} sampel pada {fs_in} Hz")
print(f"Sampel terpotong   : {len(y_naive)} sampel pada {fs_out} Hz")
print(f"Batas Nyquist baru : {fs_out / 2} Hz")

# Fungsi pembantu untuk memplot spektrum frekuensi dengan FFT
def plot_spectrum(sig, fs, title, ax, color='#0ea5e9'):
    N = len(sig)
    freqs = np.fft.rfftfreq(N, 1 / fs)
    magnitude = np.abs(np.fft.rfft(sig)) / (N / 2)
    ax.plot(freqs, magnitude, color=color, lw=1.6)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Frekuensi (Hz)', fontsize=9)
    ax.set_ylabel('Magnitudo', fontsize=9)
    ax.set_xlim(0, fs / 2)
    ax.axvline(fs / 2, color='#ef4444', linestyle='--', alpha=0.8, label=f'Batas Nyquist ({fs/2:.0f} Hz)')
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(loc='upper right', fontsize=8)

fig, axes = plt.subplots(2, 1, figsize=(9, 5))
plot_spectrum(x, fs_in, "Spektrum Sinyal Asli (fs = 8000 Hz) — Puncak Terpisah di 500 Hz & 3500 Hz", axes[0], color='#0284c7')
plot_spectrum(y_naive, fs_out, "Spektrum Downsampling Naif [::2] (fs = 4000 Hz) — Terjadi ALIASING di 500 Hz!", axes[1], color='#dc2626')
plt.tight_layout()
plt.show()

print("🔍 ANALISIS HASIL:")
print("- Komponen 3.500 Hz melipat balik tepat ke posisi: |3500 - 4000| = 500 Hz.")
print("- Akibatnya, magnitudo di frekuensi 500 Hz melonjak menjadi 1.0 (0.5 + 0.5)!")
print("- Sinyal asli kini mengalami distorsi permanen yang merusak integritas informasi.")


## 5. Solusi Standar DSP: Filter Anti-Aliasing

### 🛡️ Aturan Emas Downsampling:
> **Selalu terapkan Low-Pass Filter (LPF) SEBELUM sampel dibuang!**
>
> Frekuensi cutoff filter LPF harus memenuhi:
> $$f_{\text{cut}} \le \frac{f_{\text{target}}}{2} = \frac{f_s}{2M}$$

**Analogi Saringan Dapur:**  
Saring kotoran kasar terlebih dahulu sebelum tepung dituangkan ke mangkuk adonan. Jangan buang wadahnya dulu baru menyaring!

Mari kita bandingkan tiga pendekatan implementasi:
1. **Pendekatan Naif**: `x[::M]` (Tanpa filter $\to$ rusak parah akibat aliasing).
2. **Pendekatan Manual DSP**: Butterworth Low-Pass Filter $\to$ Slicing desimasi.
3. **Standar Industri**: `scipy.signal.resample_poly` dan `librosa.resample` (menggunakan Polyphase FIR Filter yang sangat efisien dan otomatis terlindungi anti-alias).


In [ ]:
# 1. Implementasi Manual: Butterworth Low-Pass Filter IIR
nyq_target = fs_out / 2  # 2000 Hz
cutoff = 1800  # Cutoff aman di bawah batas Nyquist target
b, a = signal.butter(N=8, Wn=cutoff / (fs_in / 2), btype='low')

# Terapkan filter ke sinyal SEBELUM membuang sampel
x_filtered = signal.filtfilt(b, a, x)
y_filtered = x_filtered[::M]

# 2. Implementasi Standar Industri: scipy.signal.resample_poly
y_poly = signal.resample_poly(x, up=1, down=M)

# 3. Implementasi Librosa
y_librosa = librosa.resample(x, orig_sr=fs_in, target_sr=fs_out)

# Visualisasi komparasi ketiga metode
fig, axes = plt.subplots(3, 1, figsize=(9, 7))
plot_spectrum(y_naive, fs_out, "1. Downsampling Naif x[::2] (RUSAK: Aliasing di 500 Hz)", axes[0], color='#dc2626')
plot_spectrum(y_filtered, fs_out, "2. Butterworth LPF + Desimasi (BERSIH: Komponen 3500 Hz teredam)", axes[1], color='#059669')
plot_spectrum(y_poly, fs_out, "3. scipy.signal.resample_poly (STANDAR DSP: Akurat & Optimal)", axes[2], color='#7c3aed')
plt.tight_layout()
plt.show()


## 6. Mekanisme Upsampling (Interpolasi Faktor $L$)

**Upsampling** adalah proses menaikkan laju sampel dengan faktor bulat $L$:
$$f_{s2} = L \cdot f_{s1}$$

Mekanisme upsampling terdiri dari **dua tahap berurutan**:

```
Sinyal Asli (fs1) 
       ↓
Tahap 1: Zero-Stuffing (Sisipkan L-1 angka nol di antara sampel asli)
       ↓  [Menghasilkan Spectral Imaging / distorsi desis tinggi]
Tahap 2: Filter Rekonstruksi (Low-Pass Filter Interpolasi dengan Gain × L)
       ↓
Sinyal Upsampled Bersih (fs2)
```

Berikut kita bedah masing-masing tahapan dalam cell terpisah beserta visualisasi preview waveformnya.


In [ ]:
# Langkah 0: Sintesis Sinyal Asli (fs1 = 4000 Hz, durasi 10 ms)
L = 3  # Target: menaikkan laju sampel 3x lipat
fs1 = 4000
t1 = np.linspace(0, 0.01, int(fs1 * 0.01), endpoint=False)
f_wave = 300  # Gelombang sinus 300 Hz
x_up = np.sin(2 * np.pi * f_wave * t1)

print(f"Laju sampel awal (fs1) : {fs1} Hz")
print(f"Jumlah sampel awal     : {len(x_up)} sampel (dalam jendela 10 ms)")

# Visualisasi Waveform Diskrit Sinyal Asli
fig, ax = plt.subplots(figsize=(8, 2.2))
ax.stem(t1 * 1000, x_up, linefmt='#0284c7', markerfmt='C0o', basefmt='k-')
ax.set_title("Langkah 0: Sinyal Asli (fs1 = 4000 Hz, 40 sampel)", fontsize=10, fontweight='bold')
ax.set_xlabel("Waktu (ms)", fontsize=9)
ax.set_ylabel("Amplitudo", fontsize=9)
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


### 🔼 Tahap 1: Zero-Stuffing (Penyisipan Nol)

**Cara Kerja:**
- Sisipkan $(L-1)$ buah angka nol di antara setiap sampel asli.
- Untuk $L = 3$, kita menyisipkan **$3 - 1 = 2$ angka nol** di antara setiap sampel:
  $$[x_0, x_1, x_2, \dots] \quad \longrightarrow \quad [x_0, \mathbf{0}, \mathbf{0}, x_1, \mathbf{0}, \mathbf{0}, x_2, \mathbf{0}, \mathbf{0}, \dots]$$
- Laju sampel baru naik menjadi: $f_{s2} = L \cdot f_{s1} = 3 \times 4.000 = 12.000\text{ Hz}$.

*Dampak:* Sinyal memiliki resolusi waktu lebih rapat, tetapi bentuknya menjadi diskrit patah-patah (banyak lembah nol) dan memunculkan duplikasi spektrum frekuensi tinggi (*spectral imaging* atau suara desis dengung tajam).


In [ ]:
# Tahap 1: Zero-Stuffing (Sisipkan L-1 = 2 angka nol antar sampel)
fs2 = fs1 * L  # 12.000 Hz
x_zero_stuffed = np.zeros(len(x_up) * L)
x_zero_stuffed[::L] = x_up  # Tempatkan sampel asli pada kelipatan L

print(f"Laju sampel baru (fs2) : {fs2} Hz")
print(f"Jumlah sampel baru     : {len(x_zero_stuffed)} sampel")
print("\nInspeksi 12 Elemen Pertama (Terlihat pola sampel asli diselingi 2 angka nol):")
print(np.round(x_zero_stuffed[:12], 3))

# Visualisasi Waveform Tahap 1: Zero-Stuffing
t_dense = np.linspace(0, 0.01, len(x_zero_stuffed), endpoint=False)

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.stem(t_dense * 1000, x_zero_stuffed, linefmt='#dc2626', markerfmt='C3^', basefmt='k-')
ax.set_title("Tahap 1: Hasil Zero-Stuffing (L = 3, 120 sampel dengan lembah angka nol)", fontsize=10, fontweight='bold')
ax.set_xlabel("Waktu (ms)", fontsize=9)
ax.set_ylabel("Amplitudo", fontsize=9)
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


### 🪄 Tahap 2: Filter Rekonstruksi (Anti-Imaging Low-Pass Filter)

**Cara Kerja:**
- Terapkan Low-Pass Filter (LPF) dengan frekuensi cutoff:
  $$f_{\text{cut}} \le \frac{f_{s1}}{2}$$
- **Fungsi Filter:**
  1. Menghaluskan transisi kurva (*connect-the-dots*) dan menginterpolasi nilai di antara sampel asli.
  2. Membuang duplikasi bayangan frekuensi tinggi (*anti-imaging*).
- **Penguatan Gain $\times L$:** Karena kita menyisipkan $(L-1)$ angka nol, energi rata-rata sinyal turun sebesar faktor $1/L$. Oleh karena itu, filter interpolasi harus diberi penguatan (*gain*) sebesar **$L$** agar amplitudo sinyal kembali seperti semula.


In [ ]:
# Tahap 2: Filter Rekonstruksi (LPF Anti-Imaging) dengan Penguatan Gain = L
nyq_new = fs2 / 2  # 6000 Hz
cutoff_up = (fs1 / 2) * 0.9  # Cutoff aman di bawah batas Nyquist awal (1800 Hz)
b_up, a_up = signal.butter(N=6, Wn=cutoff_up / nyq_new, btype='low')

# Lakukan pemfilteran dan kalikan dengan gain L
x_interpolated = L * signal.filtfilt(b_up, a_up, x_zero_stuffed)

# Visualisasi Waveform Tahap 2: Sinyal Terinterpolasi Halus vs Titik Asli
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(t_dense * 1000, x_interpolated, color='#059669', lw=2.2, label='Hasil Rekonstruksi LPF (fs2 = 12 kHz)')
ax.stem(t1 * 1000, x_up, linefmt='C0:', markerfmt='C0o', basefmt='k-', label='Sampel Asli (fs1 = 4 kHz)')
ax.set_title("Tahap 2: Hasil Akhir Upsampling (Kurva Halus Menghubungkan Sampel Asli)", fontsize=10, fontweight='bold')
ax.set_xlabel("Waktu (ms)", fontsize=9)
ax.set_ylabel("Amplitudo", fontsize=9)
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


## 7. Resampling Rasio Pecahan ($L/M$) — Kasus Nyata 44.1 kHz ke 48 kHz

Di dunia broadcasting, perfilman, dan video editing, konversi antara audio CD (44.1 kHz) dan format video profesional YouTube/TV (48 kHz) sangat umum terjadi.
Karena $48.000$ bukan kelipatan bulat dari $44.100$, rasionya disederhanakan menjadi pecahan bilangan bulat terkecil:
$$\frac{f_{s2}}{f_{s1}} = \frac{48.000}{44.100} = \frac{160}{147}$$

### Rantai Konversi Standar Industri:
1. **Upsample faktor $L = 160$** (sisipkan 159 nol antar sampel).
2. **Satu Filter LPF Tunggal** dengan cutoff $\approx 22.050\text{ Hz}$ (berfungsi ganda: *anti-imaging* untuk tahap $L=160$ sekaligus *anti-aliasing* untuk tahap $M=147$).
3. **Downsample faktor $M = 147$** (ambil 1 sampel tiap 147).

Menggunakan `scipy.signal.resample_poly` atau `librosa.resample`, komputasi ini dioptimalkan melalui *Polyphase Filter Bank* sehingga tidak perlu menyimpan array raksasa di memori.


In [ ]:
# Kasus Nyata: Konversi CD 44.1 kHz ke Video 48 kHz
fs_cd = 44100
fs_video = 48000

# Sinyal uji 1 detik (dua nada harmonik: 440 Hz dan 1000 Hz)
t_cd = np.linspace(0, 1.0, fs_cd, endpoint=False)
y_cd = 0.4 * np.sin(2 * np.pi * 440 * t_cd) + 0.3 * np.sin(2 * np.pi * 1000 * t_cd)

# Rasio pecahan paling sederhana: 160 / 147
L_frac = 160
M_frac = 147

# 1. Menggunakan scipy.signal.resample_poly
y_video_poly = signal.resample_poly(y_cd, up=L_frac, down=M_frac)

# 2. Menggunakan librosa.resample
y_video_librosa = librosa.resample(y_cd, orig_sr=fs_cd, target_sr=fs_video)

print(f"Sampel awal (44.100 Hz)   : {len(y_cd)} sampel")
print(f"Sampel target teoritis    : {int(len(y_cd) * 48000 / 44100)} sampel")
print(f"Hasil resample_poly       : {len(y_video_poly)} sampel")
print(f"Hasil librosa.resample    : {len(y_video_librosa)} sampel")


## 8. Demonstrasi Audio Nyata: Chirp Signal (Sapuan Frekuensi)

Untuk mengamati dan **mendengar** fenomena aliasing secara dramatis, kita membuat **Chirp Signal** (sinyal sapuan frekuensi yang frekuensinya terus merangkak naik secara linier dari $100\text{ Hz}$ hingga $7.500\text{ Hz}$ pada laju awal $f_{s1} = 16.000\text{ Hz}$).

Jika sinyal ini kita downsample $2\times$ ($M = 2$) ke laju target $f_{s2} = 8.000\text{ Hz}$ (Batas Nyquist baru $= 4.000\text{ Hz}$):
- **Downsample Naif `[::2]` (Tanpa Filter Anti-Aliasing):**  
  Ketika frekuensi chirp melewati batas Nyquist $4.000\text{ Hz}$, frekuensi tersebut akan **melipat balik** (*aliasing foldback*) turun dari $4.000\text{ Hz}$ menuju $8000 - 7500 = 500\text{ Hz}$!  
  Hal ini menghasilkan pola spektrogram berbentuk huruf **V** yang sangat khas dan nada suara yang terdengar **naik lalu berbalik turun**!
- **Downsample Standar DSP (Dengan Filter Polyphase Anti-Aliasing):**  
  Frekuensi di atas $4.000\text{ Hz}$ diredam bersih oleh filter sebelum desimasi. Di spektrogram dan secara auditif, nada naik hingga $4.000\text{ Hz}$ lalu **berhenti dengan bersih/hening**, tanpa ada suara pantulan sama sekali!

*(💡 **Catatan Kompatibilitas Pemutar:** Laju target $8.000\text{ Hz}$ dipilih karena standar pemutar audio peramban web/HTML5 memerlukan sample rate $\ge 8.000\text{ Hz}$ agar berkas audio dapat diputar secara lancar tanpa mengalami masalah playback).*


In [ ]:
# Sintesis sinyal chirp 3 detik (100 Hz s.d. 7500 Hz pada fs = 16.000 Hz)
fs_chirp = 16000
dur_chirp = 3.0
t_chirp = np.linspace(0, dur_chirp, int(fs_chirp * dur_chirp), endpoint=False)
y_chirp = signal.chirp(t_chirp, f0=100, t1=dur_chirp, f1=7500, method='linear')

# Downsample 2x (Laju target = 8000 Hz, Batas Nyquist target = 4000 Hz)
M_chirp = 2
fs_target_chirp = fs_chirp // M_chirp  # 8000 Hz

# Metode 1: Naif (slicing [::2] tanpa filter)
y_chirp_naive = y_chirp[::M_chirp]

# Metode 2: Standar DSP (Polyphase FIR Anti-Aliasing Filter)
y_chirp_clean = signal.resample_poly(y_chirp, up=1, down=M_chirp)

# Visualisasi Spektrogram STFT Komparasi
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

D_naive = np.abs(librosa.stft(y_chirp_naive, n_fft=512, hop_length=128))
librosa.display.specshow(librosa.amplitude_to_db(D_naive, ref=np.max), sr=fs_target_chirp, 
                         x_axis='time', y_axis='hz', ax=axes[0], cmap='magma')
axes[0].set_title("1. Downsampling Naif [::2] (Pola 'V' Akibat Aliasing)", fontweight='bold', fontsize=10)
axes[0].set_ylim(0, 4000)

D_clean = np.abs(librosa.stft(y_chirp_clean, n_fft=512, hop_length=128))
librosa.display.specshow(librosa.amplitude_to_db(D_clean, ref=np.max), sr=fs_target_chirp, 
                         x_axis='time', y_axis='hz', ax=axes[1], cmap='magma')
axes[1].set_title("2. Downsampling Standar DSP (Bersih Tanpa Pantulan)", fontweight='bold', fontsize=10)
axes[1].set_ylim(0, 4000)

plt.tight_layout()
plt.show()


In [ ]:
# Dengarkan perbedaan ketiga sinyal secara auditif:
# (Ketiga audio dapat diputar lancar di peramban karena laju sampel >= 8.000 Hz)

print("1. 🎵 Audio Chirp Asli (fs = 16.000 Hz) — Frekuensi naik konsisten dari 100 Hz s.d. 7.500 Hz:")
display(ipd.Audio(y_chirp, rate=fs_chirp))

print("2. ⚠️ Downsample Naif [::2] (fs = 8.000 Hz) — DENGARKAN: Nada memantul balik turun saat melewati 4.000 Hz!")
display(ipd.Audio(y_chirp_naive, rate=fs_target_chirp))

print("3. ✅ Downsample Standar DSP (fs = 8.000 Hz) — DENGARKAN: Nada naik ke 4.000 Hz lalu hening bersih tanpa pantulan!")
display(ipd.Audio(y_chirp_clean, rate=fs_target_chirp))


## 9. Rangkuman & Kesimpulan

1. **Sample Rate Conversion (SRC)** bukan sekadar mengganti label metadata pada file audio. Mengubah laju sampel membutuhkan rekonstruksi titik-titik sampel secara matematis.
2. **Efek Chipmunk dan Monster** terjadi jika pemutar audio membaca sampel data dengan laju yang salah tanpa melakukan proses resampling terlebih dahulu.
3. **Aliasing** terjadi saat frekuensi sinyal melanggar kriteria Nyquist ($f > f_s / 2$). Akibatnya, komponen frekuensi tinggi menyamar menjadi frekuensi rendah palsu (*spectral fold-back*).
4. **Downsampling (Desimasi $M$)** membuang $(M-1)$ sampel untuk efisiensi memori dan kecepatan inferensi model AI. Syarat mutlaknya adalah menerapkan **Filter Anti-Aliasing (LPF)** sebelum desimasi untuk mencegah kerusakan sinyal akibat aliasing.
5. **Upsampling (Interpolasi $L$)** menaikkan resolusi waktu sinyal dengan menyisipkan $(L-1)$ angka nol (*Zero-Stuffing*), lalu menghaluskan transisi kurva menggunakan **Filter Rekonstruksi (Anti-Imaging)** dengan penguatan gain sebesar $L$.
6. **Resampling Rasio Pecahan ($L/M$)** menggabungkan upsampling dan downsampling dengan satu filter LPF tunggal di tengah menggunakan arsitektur *Polyphase Filter Bank*.
7. Selalu gunakan pustaka standar industri seperti `scipy.signal.resample_poly` atau `librosa.resample` untuk hasil yang optimal, cepat, dan bebas dari distorsi.

---
© IF25-40305 Sistem Teknologi Multimedia — Institut Teknologi Sumatera (ITERA).
